<a href="https://colab.research.google.com/github/emilianomtz0189/pld-monitoreo-transaccional/blob/main/notebooks/02_reglas_monitoreo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Reglas de monitoreo transaccional

**Objetivo:** construir las reglas de alertamiento que usaría un área de PLD, medir cuánto trabajo generan y cuánto riesgo cubren.

Cada regla responde a un riesgo concreto:

| Regla | Riesgo que cubre | Tipologías que debería detectar |
|---|---|---|
| R1. Efectivo de monto alto | Introducir efectivo de origen ilícito al sistema | Depósitos o retiros grandes en efectivo |
| R2. Fraccionamiento | Dividir el efectivo para no rebasar umbrales de reporte | *Smurfing*, *Structuring* |
| R3. Concentración (fan-in / fan-out) | Juntar fondos de muchas cuentas o dispersarlos a muchas | *Fan_In*, *Fan_Out*, *Gather-Scatter* |
| R4. Movimiento rápido | Cuentas puente: el dinero entra y sale casi completo | *Deposit-Send*, *Layered*, *Cycle* |
| R5. Jurisdicciones de alto riesgo | Operaciones con países con deficiencias en PLD (listas del GAFI) | Esquemas transfronterizos |
| R6. Cambio de comportamiento | El cliente se aparta de su propio historial | *Behavioural_Change*, *Single_large* |

> **Principio de diseño:** las reglas se calibran **sin ver la etiqueta**, con referencias regulatorias, listas oficiales y percentiles del comportamiento. La etiqueta solo se usa al final para evaluarlas. La columna `tipologia` se usa únicamente para el análisis por tipología.

## Marco regulatorio de referencia (México)

Los datos son sintéticos y de Reino Unido, así que las reglas son **ilustrativas**: toman conceptos de la regulación mexicana, pero sus umbrales no sustituyen los de las disposiciones vigentes.

| Concepto | Qué establece | Cómo se adapta aquí |
|---|---|---|
| **Art. 115 de la Ley de Instituciones de Crédito** | Obliga a los bancos a establecer medidas para prevenir y detectar operaciones con recursos de procedencia ilícita, y a reportarlas. Las *Disposiciones de carácter general* las emite la SHCP; la CNBV supervisa su cumplimiento y la UIF recibe y analiza los reportes. | Marco general del proyecto. |
| **Operación Relevante** | Operación con billetes y monedas metálicas por un monto igual o superior al equivalente a **7,500 dólares** de EE. UU. (Disposición 2ª, definición de Operación Relevante). Se reporta por umbral, aunque no haya nada sospechoso. | **R1:** efectivo ≥ 7,500 USD convertidos a libras con un tipo de cambio supuesto. |
| **Operación Inusual** | Operación, actividad, conducta o comportamiento de un cliente que **no concuerda con sus antecedentes** o perfil (por monto, frecuencia, tipo de operación, origen o destino de los recursos). | **R2 a R6:** cada regla detecta un tipo de comportamiento fuera de lo esperado. Una alerta **no** es un reporte: el analista la dictamina y decide si se reporta. |
| **Operación Interna Preocupante** | Conducta de directivos, funcionarios o empleados de la institución que podría contravenir las disposiciones. | No aplica: el dataset no tiene información de empleados. |
| **Países de alto riesgo** | Las instituciones deben considerar como factor de riesgo los países con deficiencias en sus regímenes de PLD, como los que señala el GAFI. | **R5:** países del dataset en las listas del GAFI, según la fecha de cada operación. |

**Verificar en la fuente oficial (DOF):** la numeración exacta de las fracciones y la versión vigente de las Disposiciones, la periodicidad y los plazos de envío de cada reporte, y las fechas de las plenarias del GAFI.

## 1. Entorno

In [ ]:
import os
import sys
from pathlib import Path

USUARIO_GITHUB = "emilianomtz0189"
NOMBRE_REPO = "pld-monitoreo-transaccional"
CARPETA_DRIVE = Path("/content/drive/MyDrive/pld-monitoreo-transaccional")

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    ruta_repo = Path("/content") / NOMBRE_REPO
    if not ruta_repo.exists():
        !git clone -q https://github.com/{USUARIO_GITHUB}/{NOMBRE_REPO}.git {ruta_repo}
    else:
        !git -C {ruta_repo} pull -q
    os.chdir(ruta_repo)
else:
    CARPETA_DRIVE = None
    if Path.cwd().name == "notebooks":
        os.chdir("..")

sys.path.insert(0, str(Path.cwd()))
print("Directorio de trabajo:", Path.cwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from src.datos import preparar_datos, respaldar_archivo, respaldar_resultados
from src.eda import (agregar_banderas, graficar_alertas_por_dia, graficar_precision_recall,
                     graficar_recall_tipologia, guardar_figura, guardar_metricas)
from src import reglas
from src.reglas import evaluar_regla, recall_por_tipologia

config.fijar_semilla()
config.crear_carpetas()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

METRICAS = {"parametros": {}}
MARCAS = {}      # Transacciones marcadas por cada regla
RESUMEN = []     # Métricas de cada regla
ALERTAS = []     # Alertas (regla + cuenta + semana)

## 2. Datos

Se carga el Parquet respaldado en Drive en la etapa 1 (segundos). Se agregan las banderas geográficas y tres columnas de tiempo: `segundo`, `dia` y `semana`, contadas desde el primer día de los datos.

⏱️ Todo el notebook tarda alrededor de **5 a 10 minutos** con los 9.5 millones de transacciones. Las reglas usan operaciones vectorizadas para no saturar la memoria de Colab.

In [ ]:
df = preparar_datos(CARPETA_DRIVE)
df = agregar_banderas(df)
df = reglas.agregar_tiempo(df)

TOTAL_SOSPECHOSAS = int(df[config.COLUMNA_OBJETIVO].sum())
DIAS = int(df["dia"].max()) + 1
print(f"Transacciones: {len(df):,} | Sospechosas: {TOTAL_SOSPECHOSAS:,} | Días: {DIAS} | Semanas: {df['semana'].nunique()}")
print(f"Memoria: {df.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

## 3. Cómo se mide una regla

| Métrica | Pregunta de negocio |
|---|---|
| **Alertas** | ¿Cuántos casos tendría que revisar el equipo? Una alerta = regla + cuenta + semana. |
| **Alertas por día** | ¿Alcanza el equipo? Es la carga de trabajo diaria. |
| **Precisión de alertas** | De los casos revisados, ¿qué % tenía algo sospechoso? El resto son falsos positivos. |
| **Precisión por transacción** | De las transacciones marcadas, ¿qué % eran sospechosas? |
| **Recall** | De **todas** las transacciones sospechosas, ¿qué % marcó la regla? Lo que no marca, se escapa. |
| **Recall por tipología** | ¿Qué esquemas de lavado detecta y cuáles se le escapan? |

Ninguna regla sola detecta todo. Un sistema de monitoreo combina varias reglas, cada una especializada en un riesgo.

In [ ]:
def registrar(nombre, marcas, parametros):
    # Evalúa la regla, guarda sus resultados y muestra sus métricas.
    metricas, alertas = evaluar_regla(df, marcas, nombre)
    MARCAS[nombre] = marcas
    RESUMEN.append(metricas)
    ALERTAS.append(alertas)
    METRICAS["parametros"][nombre] = parametros
    return pd.Series(metricas, dtype=object).to_frame("valor")

## 4. R1 · Efectivo de monto alto

**Riesgo:** el efectivo es la puerta de entrada del dinero ilícito al sistema financiero, porque no deja rastro de su origen.

**Regla:** depósito o retiro en efectivo **igual o mayor** al equivalente de 7,500 USD en libras. El tipo de cambio es un supuesto ilustrativo.

> Esta regla es el análogo de la **operación relevante**: en México se reporta por umbral, aunque no haya nada sospechoso. Por eso no se espera que tenga alta precisión; su función es cumplir la obligación de reporte.

In [ ]:
print(f"Umbral: {reglas.UMBRAL_USD_RELEVANTE:,} USD / {reglas.USD_POR_GBP} USD por libra = £{reglas.UMBRAL_EFECTIVO_GBP:,}")
t0 = time.time()
r1 = reglas.regla_efectivo_alto(df)
print(f"Calculada en {time.time() - t0:.0f} s")
registrar("R1 Efectivo alto", r1, {"umbral_gbp": reglas.UMBRAL_EFECTIVO_GBP, "usd_por_gbp": reglas.USD_POR_GBP})

## 5. R2 · Fraccionamiento

**Riesgo:** quien sabe que existe un umbral de reporte divide el efectivo en depósitos más chicos para no rebasarlo. En México esta conducta se conoce como **fraccionamiento** y es un indicador clásico de operación inusual.

**Regla:** **3 o más** depósitos en efectivo **debajo** del umbral, en **7 días**, que **juntos** lo rebasan. Se revisa desde quien deposita y desde quien recibe, porque en el pitufeo (*smurfing*) varias personas depositan a una misma cuenta.

Se prueban dos versiones:
- **R2a "justo debajo":** solo depósitos entre el 80 % y el 100 % del umbral (la definición de libro).
- **R2b acumulado:** cualquier depósito debajo del umbral.

In [ ]:
t0 = time.time()
r2a = reglas.regla_fraccionamiento(df, piso_pct=0.8)
registrar("R2a Fraccionamiento justo debajo", r2a,
          {"umbral_gbp": reglas.UMBRAL_EFECTIVO_GBP, "dias": 7, "minimo_operaciones": 3, "piso_pct": 0.8})
r2b = reglas.regla_fraccionamiento(df, piso_pct=0.0)
registrar("R2b Fraccionamiento acumulado", r2b,
          {"umbral_gbp": reglas.UMBRAL_EFECTIVO_GBP, "dias": 7, "minimo_operaciones": 3, "piso_pct": 0.0})
print(f"Calculadas en {time.time() - t0:.0f} s")
pd.DataFrame(RESUMEN[-2:]).set_index("regla").T

## 6. R3 · Concentración (fan-in y fan-out)

**Riesgo:**
- **Fan-in:** una cuenta **recibe** dinero de muchas cuentas distintas. Así se juntan fondos de varias fuentes (por ejemplo, cuentas de terceros o "mulas").
- **Fan-out:** una cuenta **envía** a muchas cuentas distintas. Así se dispersan los fondos para perderles el rastro.

**Regla:** número de **contrapartes distintas por semana**. El umbral es el **percentil 99.9** de la distribución: se alerta al 0.1 % de cuenta-semanas más concentradas. Es una calibración por comportamiento, sin usar la etiqueta.

⏱️ Es la regla más pesada (cuenta contrapartes únicas en 9.5 millones de filas): puede tardar 1 a 2 minutos.

In [ ]:
t0 = time.time()
r3_in, umbral_in, dist_in = reglas.regla_concentracion(df, "entrada")
r3_out, umbral_out, dist_out = reglas.regla_concentracion(df, "salida")
print(f"Calculadas en {time.time() - t0:.0f} s")

percentiles = [50, 90, 99, 99.9, 100]
pd.DataFrame({"fan-in (remitentes distintos)": np.percentile(dist_in, percentiles),
              "fan-out (destinatarios distintos)": np.percentile(dist_out, percentiles)},
             index=[f"p{p}" for p in percentiles]).round(0)

In [ ]:
print(f"Umbral fan-in: {umbral_in} remitentes distintos por semana | Umbral fan-out: {umbral_out} destinatarios distintos por semana")
registrar("R3a Concentración fan-in", r3_in, {"percentil": 99.9, "umbral_contrapartes": umbral_in})
registrar("R3b Concentración fan-out", r3_out, {"percentil": 99.9, "umbral_contrapartes": umbral_out})
del dist_in, dist_out
pd.DataFrame(RESUMEN[-2:]).set_index("regla").T

## 7. R4 · Movimiento rápido de fondos

**Riesgo:** una **cuenta puente** recibe dinero y lo saca casi completo de inmediato. El dinero no se queda: solo pasa por ahí para alejarse de su origen (estratificación).

**Regla:** la cuenta recibe en un día al menos el umbral (£6,048) y, ese día o el siguiente, envía o retira entre el 90 % y el 110 % de lo recibido.

In [ ]:
t0 = time.time()
r4 = reglas.regla_movimiento_rapido(df)
print(f"Calculada en {time.time() - t0:.0f} s")
registrar("R4 Movimiento rápido", r4, {"monto_minimo_gbp": reglas.UMBRAL_EFECTIVO_GBP, "tolerancia": 0.10, "dias": 2})

## 8. R5 · Jurisdicciones de alto riesgo

**Riesgo:** operar con países con deficiencias estratégicas en PLD dificulta conocer el origen y el destino de los recursos.

**Regla:** operación internacional en la que el banco de origen o de destino está en un país que, **en la fecha de la operación**, estaba en las listas del GAFI (países bajo monitoreo reforzado o sujetos a llamado a la acción). La lista cambia en cada plenaria, así que se aplica por fecha:

In [ ]:
pd.DataFrame([{"país": p, "en la lista desde": d or "antes de oct-2022", "hasta": h or "después de ago-2023"}
              for p, (d, h) in reglas.LISTA_GAFI.items()])

> Se usa la lista oficial y **no** los países con más casos sospechosos en los datos: elegir países viendo la etiqueta sería hacer trampa, porque en la vida real no se conoce la respuesta de antemano. México no está en estas listas.

In [ ]:
t0 = time.time()
r5 = reglas.regla_alto_riesgo(df)
print(f"Calculada en {time.time() - t0:.0f} s")
registrar("R5 Jurisdicción alto riesgo", r5, {"lista": {p: list(v) for p, v in reglas.LISTA_GAFI.items()}})

## 9. R6 · Cambio de comportamiento

**Riesgo:** es la esencia de la **operación inusual**: el cliente hace algo que no concuerda con su propio historial.

**Regla** (solo usa operaciones **anteriores** de la cuenta, nunca futuras):
- **Monto:** la operación es **5 veces** mayor que el monto promedio previo de la cuenta, con al menos 10 operaciones previas y monto de al menos £1,000.
- **Frecuencia:** en los últimos 7 días la cuenta hizo al menos 5 operaciones y **3 veces** más que su ritmo semanal previo, con al menos 30 días de historia.

In [ ]:
t0 = time.time()
r6 = reglas.regla_cambio_comportamiento(df)
print(f"Calculada en {time.time() - t0:.0f} s")
registrar("R6 Cambio de comportamiento", r6,
          {"factor_monto": 5, "minimo_historial": 10, "monto_minimo_gbp": 1000,
           "factor_frecuencia": 3, "minimo_frecuencia": 5, "dias": 7, "dias_historial": 30})

## 10. Comparativo de reglas

El **consolidado** une todas las reglas: si varias reglas alertan a la misma cuenta en la misma semana, el analista revisa **un solo caso**.

In [ ]:
todas = pd.concat(MARCAS.values(), ignore_index=True).drop_duplicates()
metricas_todas, alertas_todas = evaluar_regla(df, todas, "Todas las reglas (consolidado)")

resumen = pd.DataFrame(RESUMEN + [metricas_todas]).set_index("regla")
resumen.style.format({"alertas": "{:,.0f}", "alertas_por_dia": "{:,.1f}", "cuentas_alertadas": "{:,.0f}",
                      "alertas_con_sospechosa": "{:,.0f}", "precision_alertas_pct": "{:.2f}%",
                      "transacciones_marcadas": "{:,.0f}", "sospechosas_detectadas": "{:,.0f}",
                      "precision_transacciones_pct": "{:.2f}%", "recall_pct": "{:.2f}%"})

In [ ]:
fig = graficar_alertas_por_dia(resumen)
guardar_figura(fig, "08_alertas_por_dia_por_regla.png")
plt.show()

In [ ]:
fig = graficar_precision_recall(resumen)
guardar_figura(fig, "09_precision_vs_recall_reglas.png")
plt.show()

### ¿Qué detecta cada regla?
Recall por tipología: % de las transacciones de cada esquema de lavado que marca cada regla. Una fila con valores bajos en todas las columnas es un **punto ciego** del sistema.

In [ ]:
por_tipologia = recall_por_tipologia(df, {**MARCAS, "Todas": todas})
por_tipologia.style.format({c: "{:.1f}%" for c in por_tipologia.columns if c != "transacciones"} | {"transacciones": "{:,.0f}"})

In [ ]:
fig = graficar_recall_tipologia(por_tipologia)
guardar_figura(fig, "10_recall_por_tipologia.png")
plt.show()

## 11. Guardado

- `data/processed/alertas.parquet`: todas las alertas (regla + cuenta + semana), **sin la etiqueta**. Es insumo de la etapa 3 (segmentación) y la etapa 4 (modelo). Se respalda en Drive.
- `resultados/metricas/02_reglas.json`: parámetros y métricas de cada regla, fuente de cifras para el README.

In [ ]:
alertas = pd.concat(ALERTAS, ignore_index=True)[["regla", "cuenta", "semana", "transacciones"]]
alertas["regla"] = alertas["regla"].astype("category")
alertas.to_parquet(config.ARCHIVO_ALERTAS, index=False)
print(f"Alertas guardadas: {len(alertas):,} filas en {config.ARCHIVO_ALERTAS.relative_to(config.RAIZ)}")
respaldar_archivo(config.ARCHIVO_ALERTAS, CARPETA_DRIVE)

METRICAS["umbral_efectivo_gbp"] = reglas.UMBRAL_EFECTIVO_GBP
METRICAS["total_sospechosas"] = TOTAL_SOSPECHOSAS
METRICAS["dias"] = DIAS
METRICAS["reglas"] = resumen.reset_index().to_dict("records")
METRICAS["recall_por_tipologia"] = por_tipologia.round(2).reset_index().to_dict("records")
guardar_metricas(METRICAS, "02_reglas.json")
if CARPETA_DRIVE is not None:
    respaldar_resultados(CARPETA_DRIVE)

In [ ]:
c = metricas_todas
mejor_recall = resumen.drop(index="Todas las reglas (consolidado)")["recall_pct"].idxmax()
mejor_precision = resumen.drop(index="Todas las reglas (consolidado)")["precision_alertas_pct"].idxmax()
ciegas = por_tipologia.index[por_tipologia["Todas"] < 50].tolist()
print(f'''
RESUMEN ETAPA 2
- Consolidado: {c["alertas"]:,} alertas ({c["alertas_por_dia"]:,} por día) sobre {c["cuentas_alertadas"]:,} cuentas.
- Precisión de alertas: {c["precision_alertas_pct"]}% | Recall: {c["recall_pct"]}% ({c["sospechosas_detectadas"]:,} de {TOTAL_SOSPECHOSAS:,} sospechosas).
- Regla con mayor recall: {mejor_recall} ({resumen.loc[mejor_recall, "recall_pct"]}%).
- Regla con mayor precisión: {mejor_precision} ({resumen.loc[mejor_precision, "precision_alertas_pct"]}%).
- Tipologías con menos de 50% detectado por el conjunto de reglas: {", ".join(ciegas) if ciegas else "ninguna"}.
''')

**Siguiente paso:** etapa 3, segmentación de clientes por riesgo, y etapa 4, modelo que prioriza las alertas para revisar primero las más riesgosas.